In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression as lr 
import matplotlib.pyplot as plt
import statsmodels.api as sm
from itertools import combinations

## Data:

In [ ]:
# Load Data:
path_train = r"C:\Users\oskar\OneDrive\Desktop\4 Semester\Dataproject\Federated-dental-risk-vol2\federated-dental-risk-prediction\data\raw\synthetic_dataset_A_non-iid.csv"
df = pd.read_csv(path_train)

path_test = r"C:\Users\oskar\OneDrive\Desktop\4 Semester\Dataproject\Federated-dental-risk-vol2\federated-dental-risk-prediction\data\processed\A\global_test_set_non-iid.csv"
test_data = pd.read_csv(path_test)

# Define variabels:
variables = df.columns[2:27].tolist()

X_train = df[variables].copy()
X_test = test_data[variables].copy()

# Function for interaktions:
def add_interactions(X, variables):
    interaction_cols = {}

    for v1, v2 in combinations(variables, 2):
        interaction_cols[f"{v1}_x_{v2}"] = X[v1] * X[v2]

    interaction_df = pd.DataFrame(interaction_cols, index=X.index)

    return pd.concat([X, interaction_df], axis=1)

#Add interactions
train_design_df = add_interactions(X_train, variables)
test_design_df = add_interactions(X_test, variables)

# Sørg for samme kolonner (meget vigtigt!)
test_design_df = test_design_df[train_design_df.columns]


# Standardise
scaler = StandardScaler()

train_design = pd.DataFrame(
    scaler.fit_transform(train_design_df),
    columns=train_design_df.columns,
    index=train_design_df.index
)

test_design = pd.DataFrame(
    scaler.transform(test_design_df),
    columns=test_design_df.columns,
    index=test_design_df.index
)

## Functions:

In [20]:
# Backward:
def backward_elimination(X, y, alpha=0.05, verbose=True):
    """
    Backward elimination baseret på p-værdier.
    
    Parametre
    ---------
    X : pd.DataFrame
        Forklarende variable.
    y : pd.Series eller array-lignende
        Responsvariabel.
    alpha : float
        Signifikansniveau. Variable med p-værdi > alpha fjernes.
    verbose : bool
        Hvis True, printes hvilke variable der fjernes undervejs.
    
    Returnerer
    ----------
    selected_features : list
        Navne på de features der er tilbage.
    final_model : statsmodels-regression-resultat
        Den endelige fitted model.
    """
    
    X_selected = X.copy()
    
    while True:
        X_with_const = sm.add_constant(X_selected, has_constant="add")
        model = sm.OLS(y, X_with_const).fit()
        
        p_values = model.pvalues.drop("const", errors="ignore")
        max_p_value = p_values.max()
        
        if max_p_value > alpha:
            feature_to_remove = p_values.idxmax()
            if verbose:
                print(f"Fjerner '{feature_to_remove}' med p-værdi = {max_p_value:.4f}")
            X_selected = X_selected.drop(columns=[feature_to_remove])
        else:
            break
    
    final_X = sm.add_constant(X_selected, has_constant="add")
    final_model = sm.OLS(y, final_X).fit()
    
    return list(X_selected.columns), final_model

def forward_selection(X, y, alpha=0.05, verbose=True):
    """
    Forward selection baseret på p-værdier.

    Parametre
    ---------
    X : pd.DataFrame
        Forklarende variable.
    y : pd.Series eller array-lignende
        Responsvariabel.
    alpha : float
        Signifikansniveau. En variabel tilføjes kun hvis p-værdien < alpha.
    verbose : bool
        Hvis True, printes hvilke variable der tilføjes undervejs.

    Returnerer
    ----------
    selected_features : list
        Navne på de features der er valgt.
    final_model : statsmodels-regression-resultat
        Den endelige fitted model.
    """

    remaining_features = list(X.columns)
    selected_features = []

    while remaining_features:
        pvals = {}

        for feature in remaining_features:
            features_to_test = selected_features + [feature]
            X_model = sm.add_constant(X[features_to_test], has_constant="add")
            model = sm.OLS(y, X_model).fit()
            pvals[feature] = model.pvalues[feature]

        best_feature = min(pvals, key=pvals.get)
        best_pval = pvals[best_feature]

        if best_pval < alpha:
            selected_features.append(best_feature)
            remaining_features.remove(best_feature)
            if verbose:
                print(f"Tilføjer '{best_feature}' med p-værdi = {best_pval:.4f}")
        else:
            break

    final_X = sm.add_constant(X[selected_features], has_constant="add")
    final_model = sm.OLS(y, final_X).fit()

    return selected_features, final_model

def multiple_regression(df, variables, target):
    '''
    Input: df (dataframe), variables (all the variables you want to include in the regression as a list)
    Returns: (intercept, [list of coefficients])
    '''
    n = len(df)
    d = len(variables)

    # construct empty design matrix
    X = np.zeros((n, d))

    # targets
    y = target

    # fill out design matrix 
    for i in range(d):
        X[:, i] = df[variables[i]]
    #make model
    model = lr()
    #fit model
    model.fit(X,  y)
    return model.intercept_, model.coef_

def metrics_for_one_complication(y_true, y_pred):
    """
    complication fx 'Risk_AlveolarOsteitis'
    
    Bruger:
      true: Risk_Category_AlveolarOsteitis
      pred: Risk_AlveolarOsteitis_group
    """
    
    results = {}
    
    for cls in [0, 1, 2]:
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        
        results[cls] = {
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }
    
    macro_f1 = np.mean([results[cls]["f1"] for cls in [0, 1, 2]])
    
    return {
        "per_class": results,
        "macro_f1": macro_f1
    }

def predict_from_coefs(df, variables, intercept, coefs):
    """
    Beregner y-hat = intercept + X beta
    for alle rækker i df.
    """
    X = df[variables].to_numpy()
    y_hat = intercept + X @ coefs
    return y_hat

def assign_tertiles(y_hat):
    """
    Deler y_hat i 3 grupper baseret på 1/3 og 2/3 kvantiler.
    Returnerer både gruppe-labels og grænserne.
    """
    q1 = np.quantile(y_hat, 1/3)
    q2 = np.quantile(y_hat, 2/3)

    groups = np.where(
        y_hat <= q1, 0,
        np.where(y_hat <= q2, 1, 2)
    )

    return groups, q1, q2

## Results:

### Backward Centralized:

In [24]:
# We reduce the training data, otherwise the computation is heavy:
design_df = train_design[:2500]

features, model = backward_elimination(train_design, df["Risk_AlveolarOsteitis"])

inter, coef = multiple_regression(train_design, features, df["Risk_AlveolarOsteitis"])

base_variables = [
    col for col in features
    if col not in ["Patient", "Client", "Risk_AlveolarOsteitis"]
]

preds = predict_from_coefs(test_design, base_variables, inter,coef)
cat_pred, a, b = assign_tertiles(preds)

y_true = test_data["Risk_Category_AlveolarOsteitis"].values
y_pred = cat_pred

metrics_for_one_complication(y_true, y_pred)

Fjerner 'Smoking' med p-værdi = 0.9961
Fjerner 'Caries_Adjacent_x_Tooth_Mobility' med p-værdi = 0.9945
Fjerner 'Clotting_Disorder_x_Prev_Extraction_Issue' med p-værdi = 0.9875
Fjerner 'Pericoronitis_x_Root_Curvature' med p-værdi = 0.9861
Fjerner 'Bisphosphonates_x_Prev_Extraction_Issue' med p-værdi = 0.9825
Fjerner 'Root_Development_x_Clotting_Disorder' med p-værdi = 0.9774
Fjerner 'Caries_Adjacent_x_Impaction_Depth' med p-værdi = 0.9764
Fjerner 'Swelling_x_Osteoporosis' med p-værdi = 0.9745
Fjerner 'Root_Curvature_x_Osteoporosis' med p-værdi = 0.9743
Fjerner 'Swelling_x_Clotting_Disorder' med p-værdi = 0.9720
Fjerner 'Tooth_Angulation_x_Clotting_Disorder' med p-værdi = 0.9697
Fjerner 'Clotting_Disorder_x_Bisphosphonates' med p-værdi = 0.9676
Fjerner 'Sex_x_Diabetes' med p-værdi = 0.9655
Fjerner 'Root_Count_x_Clotting_Disorder' med p-værdi = 0.9652
Fjerner 'Osteoporosis_x_Clotting_Disorder' med p-værdi = 0.9654
Fjerner 'Root_Curvature_x_Bone_Density' med p-værdi = 0.9631
Fjerner 'Swell

KeyboardInterrupt: 

### Forward Centralized:

In [10]:
features, model = forward_selection(train_design, df["Risk_AlveolarOsteitis"])

Tilføjer 'Impaction_Depth_x_Proximity_Nerve' med p-værdi = 0.0000
Tilføjer 'Pericoronitis_x_Proximity_Nerve' med p-værdi = 0.0000
Tilføjer 'Age_x_Surgical_Extraction_Type' med p-værdi = 0.0000
Tilføjer 'Pericoronitis_x_Impaction_Depth' med p-værdi = 0.0000
Tilføjer 'Age_x_Proximity_Nerve' med p-værdi = 0.0000
Tilføjer 'Pericoronitis' med p-værdi = 0.0000
Tilføjer 'Trismus_x_Root_Curvature' med p-værdi = 0.0002
Tilføjer 'Caries_Adjacent_x_Root_Curvature' med p-værdi = 0.0053
Tilføjer 'Age_x_Pericoronitis' med p-værdi = 0.0083
Tilføjer 'Caries_Wisdom_x_Root_Curvature' med p-værdi = 0.0237
Tilføjer 'Root_Curvature_x_Bisphosphonates' med p-værdi = 0.0235
Tilføjer 'Impaction_Depth_x_Surgical_Extraction_Type' med p-værdi = 0.0277
Tilføjer 'Patient_x_Smoking' med p-værdi = 0.0309
Tilføjer 'Pericoronitis_x_Tooth_Mobility' med p-værdi = 0.0401
Tilføjer 'Caries_Wisdom_x_Clotting_Disorder' med p-værdi = 0.0457
Tilføjer 'Smoking_x_Bisphosphonates' med p-værdi = 0.0492


In [23]:
inter, coef = multiple_regression(train_design, features, df["Risk_AlveolarOsteitis"])

base_variables = [
    col for col in features
    if col not in ["Patient", "Client", "Risk_AlveolarOsteitis"]
]

preds = predict_from_coefs(test_design, base_variables, inter,coef)
cat_pred, a, b = assign_tertiles(preds)

y_true = test_data["Risk_Category_AlveolarOsteitis"].values
y_pred = cat_pred

metrics_for_one_complication(y_true, y_pred)

C:\Users\oskar\AppData\Local\Temp\ipykernel_15540\317289892.py:3: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  design_df_merged = train_design.merge(df[["Patient",'Risk_AlveolarOsteitis',"Client"]], on = "Patient", how='left')


{'per_class': {0: {'TP': np.int64(769),
   'FP': np.int64(231),
   'FN': np.int64(196),
   'precision': np.float64(0.769),
   'recall': np.float64(0.7968911917098446),
   'f1': np.float64(0.7826972010178117)},
  1: {'TP': np.int64(562),
   'FP': np.int64(439),
   'FN': np.int64(465),
   'precision': np.float64(0.5614385614385614),
   'recall': np.float64(0.5472249269717624),
   'f1': np.float64(0.5542406311637081)},
  2: {'TP': np.int64(729),
   'FP': np.int64(270),
   'FN': np.int64(279),
   'precision': np.float64(0.7297297297297297),
   'recall': np.float64(0.7232142857142857),
   'f1': np.float64(0.726457399103139)}},
 'macro_f1': np.float64(0.6877984104282197)}

Centralized has f1-macro = 0.59 on AlveolarOsteitis with 2500 datapunkter in the base model without interaktions.